In [ ]:
#! mlflow server -p 5001

In [ ]:
import mlflow
from openai import OpenAI
from loguru import logger

PORT = "5001"
EXPERIMENT_NAME = "Chatbot"



In [ ]:
# Set up MLflow autologging and tracking URI
mlflow.openai.autolog()
mlflow.set_tracking_uri(f"http://localhost:{PORT}")
mlflow.set_experiment(EXPERIMENT_NAME)

In [ ]:
OLLAMA_API_URL = "http://localhost:11434/v1"
LLM_MODEL = "gpt-oss:20b"
TEMPERATURE = 0.1
MAX_TOKENS = 100

class Agent:
    def __init__(self, system=""):
        self.client = self._set_llm(OLLAMA_API_URL)
        self.system = system
        self.messages = []
        if self.system:
            self.messages.append({"role": "system", "content": system})

    def _set_llm(self, url):
        try:
            logger.info(f"Ollama API URL: {url}")
            return OpenAI(
                base_url=url,
                api_key="dummy",
            )
        except Exception as e:
            print(f"Error initializing OpenAI client: {e}")
            return None

    def __call__(self, message):
        try:
            self.messages.append({"role": "user", "content": message})
            result = self.execute()
            self.messages.append({"role": "assistant", "content": result})
            return result
        except Exception as e:
            logger.error(f"Error during agent call: {e}")
            return "An error occurred while processing your request."

    def execute(self):
        try:
            response = self.client.chat.completions.create(
                        model=LLM_MODEL,
                        messages=self.messages,
                        temperature=TEMPERATURE,
                        max_tokens=MAX_TOKENS,
                    )
            return response.choices[0].message.content
        except Exception as e:
            logger.error(f"Error during agent execution: {e}")
            return "An error occurred while processing your request."

In [ ]:
agent = Agent(system="You are a helpful assistant.")
response_0 = agent("Why is the sky blue?")
response_1 = agent("What is the capital of France?")


In [ ]:
print("Response 0:", response_0)
print(" ------------------------------")
print("Response 1:", response_1)